<a href="https://colab.research.google.com/github/saikttech/colab-aiagent/blob/main/%D0%94%D0%97_Pro_%D0%9D%D0%B5%D0%B9%D1%80%D0%BE_%D0%BA%D0%BE%D0%BD%D1%81%D1%83%D0%BB%D1%8C%D1%82%D0%B0%D0%BD%D1%82_%D0%B4%D0%BE%D0%BE%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%BD%D1%8B%D0%B9_%D0%BD%D0%B0_%D0%B1%D0%B0%D0%B7%D0%B5_%D0%B7%D0%BD%D0%B0%D0%BD%D0%B8%D0%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Задание:

На основе Правил безопасности опасных производственных объектов, на которых используются подъемные сооружения (Документ для создания базы знаний: https://docs.google.com/document/d/1q4l912Re8zuIfBax4FDS3ZppYmVPzER3Si2wrmznddc), сделайте:
- нейро-консультанта для ответов на вопросы по представленному документу, проработайте промт самостоятельно;
- Проверьте работоспособность созданного консультанта на нескольких самостоятельно сформулированных вопросах.

# Решение

In [ ]:
# Нейро-консультант по ФНП подъемных сооружений на базе Yandex GPT
# Улучшенная версия с ТОЧНЫМ подсчетом токенов из API

import requests
import json
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ============================================================
# 1. ПОЛУЧЕНИЕ СЕКРЕТОВ
# ============================================================
def get_secret(name, default_value=""):
    """Получение секрета из Colab или возврат дефолтного значения"""
    try:
        return userdata.get(name)
    except userdata.SecretNotFoundError:
        return default_value
    except Exception as e:
        print(f"⚠️ Ошибка получения секрета {name}: {e}")
        return default_value

FOLDER_ID = get_secret('FOLDER_ID', 'b1gXXXXXXXXXX')
IAM_TOKEN = get_secret('IAM_TOKEN', 't1.9euelZq...')

# ============================================================
# 2. БАЗА ЗНАНИЙ (сокращенная версия для экономии токенов)
# ============================================================
KNOWLEDGE_BASE = """
ПРАВИЛА БЕЗОПАСНОСТИ ОПАСНЫХ ПРОИЗВОДСТВЕННЫХ ОБЪЕКТОВ,
НА КОТОРЫХ ИСПОЛЬЗУЮТСЯ ПОДЪЕМНЫЕ СООРУЖЕНИЯ
(Приказ Ростехнадзора от 26.11.2020 N 461)

I. ОБЩИЕ ПОЛОЖЕНИЯ

1. ФНП устанавливают требования к деятельности в области промышленной безопасности
на ОПО, на которых используются стационарно установленные грузоподъемные механизмы,
подъемные сооружения (ПС).

2. Требования распространяются на:
а) грузоподъемные краны всех типов;
б) мостовые краны-штабелеры;
в) краны-трубоукладчики;
г) краны-манипуляторы;
д) строительные подъемники;
е) подъемники (вышки) для перемещения людей;
ж) грузовые электрические тележки;
з) электрические тали;
и) краны-экскаваторы;
к) сменные грузозахватные органы и приспособления;
л) грузовую тару;
м) специальные съемные кабины и люльки;
н) рельсовые пути.

3. Требования НЕ распространяются на:
а) ПС в интересах обороны и безопасности государства;
б) ПС на объектах атомной энергии;
в) ПС с ручным приводом, лифты, канатные дороги, фуникулеры, эскалаторы,
   электро- и автопогрузчики;
г) ПС в шахтах и на плавучих средствах;
д) ПС для работы только с навесным оборудованием;
е) монтажные полиспасты;
ж) краны для затворов гидротехнических сооружений;
з) домкраты;
и) манипуляторы в технологических процессах;
к) подъемники с высотой подъема до 6 м включительно;
л) ПС для аттракционов.

ЦЕЛЬ И ПРИНЦИПЫ ПРОМЫШЛЕННОЙ БЕЗОПАСНОСТИ

8. Цель ФНП - создание организационной основы обеспечения промышленной безопасности,
направленной на предотвращение и/или минимизацию последствий аварий, инцидентов.

9. Общие принципы промышленной безопасности:
а) соответствие паспортных характеристик ПС требованиям технологического процесса;
б) соответствие группы классификации ПС требованиям технологического процесса;
в) соответствие прочности, жесткости, устойчивости элементов металлоконструкции нагрузкам;
г) соответствие оснащенности ПС регистраторами, ограничителями и указателями;
д) соответствие фактического срока службы ПС указанному изготовителем;
е) соответствие прочности строительных конструкций нагрузкам от ПС;
ж) соответствие требованиям промышленной безопасности при монтаже, эксплуатации, ремонте;
з) соответствие порядку действий при аварии или инциденте.

ТРЕБОВАНИЯ К ОРГАНИЗАЦИЯМ И РАБОТНИКАМ (МОНТАЖ, НАЛАДКА, РЕМОНТ)

10. Деятельность по монтажу, наладке, ремонту осуществляют специализированные организации.

19. Требования к работникам:
а) знать схемы и приемы монтажа, иметь удостоверение;
б) знать источники опасностей и способы защиты;
в) знать и уметь выявлять дефекты;
г) уметь выполнять наладочные работы;
д) уметь применять технологии ремонта;
е) знать такелажные приспособления;
ж) знать порядок обмена сигналами;
з) иметь документы о профессиональном обучении;
и) знать методы испытаний;
к) соблюдать требования эксплуатационных документов;
л) быть аттестованными на знание ФНП (ИТР);
м) соответствовать требованиям сварочного производства.

ТРЕБОВАНИЯ К ЭКСПЛУАТИРУЮЩИМ ОРГАНИЗАЦИЯМ

22. Эксплуатирующая организация должна:
а) поддерживать ПС в работоспособном состоянии;
б) не нарушать требования паспорта и руководства по эксплуатации;
в) не допускать неработоспособные грузозахватные приспособления;
г) не эксплуатировать ПС с неработоспособными ограничителями;
д) не эксплуатировать на неработоспособных рельсовых путях;
е) не эксплуатировать с нарушениями установки;
ж) соблюдать размеры между ПС и строительными конструкциями;
з) не допускать эксплуатацию на площадках с недостаточной нагрузочной характеристикой;
и) разработать инструкции и назначить ответственных лиц;
к) устанавливать порядок допуска к работе;
л) обеспечивать соблюдение технологических процессов;
м) не допускать транспортировку работников (кроме случаев в п. 235-247);
н) исключить подтаскивание грузов и отклонение канатов;
о) иметь грузы для испытаний;
п) обеспечивать ограждение опасных зон.

25. Требования к работникам ОПО:
а) иметь удостоверение на право самостоятельной работы;
б) знать критерии работоспособности ПС;
в) информировать руководителя об угрозе аварии;
г) знать порядок действий при авариях;
д) стропальщики должны применять отличительные знаки.

БРАКОВКА ЭЛЕМЕНТОВ ПС

274. Запрещается эксплуатация стропов с дефектами:
- при наличии совокупности дефектов на площади более 10% ширины и длины;
- при размочаливании или износе более 10% ширины петель.

275. Требования к браковке элементов:
ХОДОВЫЕ КОЛЕСА:
1. Трещины любых размеров
2. Выработка реборды более 50% от первоначальной толщины
3. Выработка поверхности катания, уменьшающая диаметр на 2%
4. Разность диаметров колес более 0,5%

КРЮКИ:
1. Трещины и надрывы на поверхности
2. Износ зева более 10% от первоначальной высоты

БАРАБАНЫ:
1. Трещины любых размеров
2. Износ ручья по профилю более 2 мм

БЛОКИ:
- Износ ручья более 40% от первоначального радиуса

ТРЕБОВАНИЯ К КАНАТАМ

Минимальные коэффициенты использования канатов:
M1: подвижные - 3,15, неподвижные - 2,50
M2: подвижные - 3,35, неподвижные - 2,50
M3: подвижные - 3,55, неподвижные - 3,00
M4: подвижные - 4,00, неподвижные - 3,50
M5: подвижные - 4,50, неподвижные - 4,00
M6: подвижные - 5,60, неподвижные - 4,50
M7: подвижные - 7,10, неподвижные - 5,00
M8: подвижные - 9,00, неподвижные - 5,00

БЕЗОПАСНЫЕ РАССТОЯНИЯ

Таблица 3. Минимальное расстояние от стрелы ПС до проводов ЛЭП:
До 1 кВ - 1,5 м
Свыше 1 до 35 кВ - 2,0 м
Свыше 35 до 110 кВ - 3,0 м
Свыше 110 до 220 кВ - 4,0 м
Свыше 220 до 400 кВ - 5,0 м
Свыше 400 до 750 кВ - 9,0 м
Свыше 750 до 1150 кВ - 10,0 м

ГРУППА КЛАССИФИКАЦИИ МЕХАНИЗМА

Классы использования механизма (T - суммарная продолжительность работы):
T1: до 200 ч
T2: св. 200 до 400 ч
T3: св. 400 до 800 ч
T4: св. 800 до 1600 ч
T5: св. 1600 до 3200 ч
T6: св. 3200 до 6300 ч
T7: св. 6300 до 12500 ч
T8: св. 12500 до 25000 ч
T9: св. 25000 до 50000 ч
T10: св. 50000 до 100000 ч

Классы нагружения:
Q1 (легкий): до 0,125
Q2 (средний): св. 0,125 до 0,250
Q3 (тяжелый): св. 0,25 до 0,50
Q4 (весьма тяжелый): св. 0,50 до 1,00
"""

# ============================================================
# 3. СИСТЕМНЫЙ ПРОМТ
# ============================================================
SYSTEM_PROMPT = """Ты - нейро-консультант по правилам безопасности опасных производственных объектов,
на которых используются подъемные сооружения (ФНП, Приказ Ростехнадзора от 26.11.2020 N 461).

ТВОЯ РОЛЬ:
- Отвечать ТОЛЬКО на основе предоставленной базы знаний
- Быть точным и конкретным в ответах
- Цитировать пункты правил, когда это возможно
- Если вопрос выходит за рамки базы знаний, честно сказать об этом

ПРАВИЛА ОТВЕТА:
1. Отвечай кратко, но полно
2. Используй нумерацию пунктов из документа
3. Если вопрос не относится к правилам безопасности ПС, вежливо откажись отвечать
4. Всегда ссылайся на конкретные пункты ФНП

БАЗА ЗНАНИЙ:
""" + KNOWLEDGE_BASE

# ============================================================
# 4. ФУНКЦИИ РАБОТЫ С API (ИСПРАВЛЕННЫЙ ПОДСЧЕТ ТОКЕНОВ)
# ============================================================
def call_yandex_gpt(user_message, temperature=0.3, max_tokens=2000):
    """Вызов Yandex GPT API с ТОЧНЫМ подсчетом токенов из ответа API"""
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {IAM_TOKEN}"
    }

    payload = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": temperature,
            "maxTokens": str(max_tokens)
        },
        "messages": [
            {"role": "system", "text": SYSTEM_PROMPT},
            {"role": "user", "text": user_message}
        ]
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()

        if 'result' in result and 'alternatives' in result['result']:
            answer = result['result']['alternatives'][0]['message']['text']

            # ✅ ТОЧНЫЙ ПОДСЧЕТ ТОКЕНОВ ИЗ API ОТВЕТА
            usage = result['result'].get('usage', {})
            input_tokens = int(usage.get('inputTextTokens', '0'))
            output_tokens = int(usage.get('completionTokens', '0'))
            total_tokens = int(usage.get('totalTokens', '0'))

            return {
                'success': True,
                'answer': answer,
                'input_tokens': input_tokens,      # Токены в промте (system + user)
                'output_tokens': output_tokens,    # Токены в ответе
                'total_tokens': total_tokens,      # Всего токенов
                'status_code': response.status_code,
                'model_version': result['result'].get('modelVersion', 'unknown')
            }
        else:
            return {
                'success': False,
                'answer': f'Ошибка формата ответа: {json.dumps(result, ensure_ascii=False, indent=2)}',
                'input_tokens': 0,
                'output_tokens': 0,
                'total_tokens': 0,
                'status_code': response.status_code,
                'model_version': 'unknown'
            }
    except requests.exceptions.RequestException as e:
        return {
            'success': False,
            'answer': f'Ошибка API: {str(e)}',
            'input_tokens': 0,
            'output_tokens': 0,
            'total_tokens': 0,
            'status_code': None,
            'model_version': 'unknown'
        }

# ============================================================
# 5. СОЗДАНИЕ ИНТЕРАКТИВНОГО ИНТЕРФЕЙСА
# ============================================================

header = widgets.HTML("""
<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 20px; border-radius: 10px; color: white; margin-bottom: 15px;'>
    <h2 style='margin: 0;'>🏗️ Нейро-консультант по ФНП подъемных сооружений</h2>
    <p style='margin: 5px 0 0 0;'>Приказ Ростехнадзора от 26.11.2020 N 461</p>
    <p style='margin: 5px 0 0 0; font-size: 12px;'>✅ Точный подсчет токенов из API</p>
</div>
""")

folder_input = widgets.Text(
    value=FOLDER_ID,
    description='FOLDER_ID:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%'),
    placeholder='b1gXXXXXXXXXX'
)

token_input = widgets.Password(
    value=IAM_TOKEN,
    description='IAM_TOKEN:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%'),
    placeholder='t1.9euelZq...'
)

test_questions = [
    "-- Выберите готовый вопрос или введите свой --",
    "На какие подъемные сооружения распространяются требования ФНП?",
    "Какие требования предъявляются к работникам, осуществляющим монтаж ПС?",
    "При каком износе крюк подлежит браковке?",
    "Какое минимальное расстояние должно быть от стрелы крана до ЛЭП напряжением 110 кВ?",
    "Какие организации могут осуществлять ремонт подъемных сооружений?",
    "На какие виды подъемных сооружений НЕ распространяются ФНП?",
    "Какие коэффициенты использования канатов для класса M5?",
    "Каковы общие принципы промышленной безопасности согласно п. 9 ФНП?",
    "Какие требования предъявляются к эксплуатирующим организациям?"
]

question_dropdown = widgets.Dropdown(
    options=test_questions,
    value=test_questions[0],
    description='📋 Готовые вопросы:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%')
)

question_input = widgets.Textarea(
    value='',
    placeholder='Введите ваш вопрос по ФНП подъемных сооружений...',
    description='❓ Ваш вопрос:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', height='100px')
)

temperature_slider = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.1,
    description='🌡️ Temperature:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%'),
    readout_format='.1f'
)

max_tokens_slider = widgets.IntSlider(
    value=2000,
    min=100,
    max=8000,
    step=100,
    description='📏 Max tokens:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

ask_button = widgets.Button(
    description='🚀 Задать вопрос',
    button_style='primary',
    icon='paper-plane',
    layout=widgets.Layout(width='200px', height='40px')
)

clear_button = widgets.Button(
    description='🗑️ Очистить',
    button_style='warning',
    icon='trash',
    layout=widgets.Layout(width='150px', height='40px')
)

output_area = widgets.Output(
    layout=widgets.Layout(
        width='100%',
        min_height='200px',
        border='1px solid #e0e0e0',
        padding='15px',
        border_radius='8px'
    )
)

history_area = widgets.Output(
    layout=widgets.Layout(
        width='100%',
        max_height='300px',
        overflow='auto',
        border='1px solid #e0e0e0',
        padding='10px'
    )
)

# ============================================================
# 6. ОБРАБОТЧИКИ СОБЫТИЙ
# ============================================================
question_history = []

def on_dropdown_change(change):
    """Обработчик выбора готового вопроса"""
    if change['new'] != test_questions[0]:
        question_input.value = change['new']

def on_ask_click(b):
    """Обработчик нажатия кнопки 'Задать вопрос'"""
    global FOLDER_ID, IAM_TOKEN

    FOLDER_ID = folder_input.value.strip()
    IAM_TOKEN = token_input.value.strip()

    question = question_input.value.strip()

    with output_area:
        clear_output()

        if not question:
            print("⚠️ Пожалуйста, введите вопрос!")
            return

        if not FOLDER_ID or not IAM_TOKEN:
            print("❌ Ошибка: FOLDER_ID и IAM_TOKEN должны быть заполнены!")
            return

        print(f"🔍 Вопрос: {question}")
        print("⏳ Ожидание ответа от Yandex GPT...")
        print()

        result = call_yandex_gpt(
            question,
            temperature=temperature_slider.value,
            max_tokens=max_tokens_slider.value
        )

        if result['success']:
            print("✅ ОТВЕТ КОНСУЛЬТАНТА:")
            print("=" * 70)
            print(result['answer'])
            print("=" * 70)
            print()
            print("📊 ТОЧНАЯ СТАТИСТИКА ТОКЕНОВ (из API):")
            print(f"  • Токенов в промте (input):  {result['input_tokens']}")
            print(f"  • Токенов в ответе (output): {result['output_tokens']}")
            print(f"  • Всего токенов (total):     {result['total_tokens']}")
            print(f"  • Temperature:               {temperature_slider.value}")
            print(f"  • Версия модели:             {result['model_version']}")

            # Добавляем в историю
            question_history.append({
                'question': question,
                'answer': result['answer'],
                'tokens': result['total_tokens'],
                'input_tokens': result['input_tokens'],
                'output_tokens': result['output_tokens']
            })
            update_history()
        else:
            print(f"❌ Ошибка (код {result['status_code']}):")
            print(result['answer'])

def on_clear_click(b):
    """Обработчик очистки"""
    question_input.value = ''
    question_dropdown.value = test_questions[0]
    with output_area:
        clear_output()

def update_history():
    """Обновление истории вопросов"""
    with history_area:
        clear_output()
        if not question_history:
            print("📭 История пуста")
            return

        print("📚 ИСТОРИЯ ЗАПРОСОВ:")
        print("-" * 60)
        for i, item in enumerate(reversed(question_history[-10:]), 1):
            print(f"{i}. ❓ {item['question'][:60]}...")
            print(f"   💬 {item['answer'][:80]}...")
            print(f"   🎯 Токенов: {item['tokens']} (input: {item['input_tokens']}, output: {item['output_tokens']})")
            print()

question_dropdown.observe(on_dropdown_change, names='value')
ask_button.on_click(on_ask_click)
clear_button.on_click(on_clear_click)

# ============================================================
# 7. ОТРИСОВКА ИНТЕРФЕЙСА
# ============================================================
display(header)

display(widgets.HTML("<h3>🔐 Настройки подключения к Yandex Cloud</h3>"))
display(folder_input)
display(token_input)
display(widgets.HTML("""
<div style='background: #fff3cd; padding: 10px; border-left: 4px solid #ffc107; margin: 10px 0;'>
    <b>💡 Подсказка:</b> Секреты загружаются из Colab Secrets (🔑 слева).
    Если они не найдены - введите значения вручную в поля выше.
</div>
"""))

display(widgets.HTML("<h3>💬 Задать вопрос</h3>"))
display(question_dropdown)
display(question_input)

settings_box = widgets.HBox([temperature_slider, max_tokens_slider])
display(widgets.HTML("<h3>⚙️ Настройки модели</h3>"))
display(settings_box)

buttons_box = widgets.HBox([ask_button, clear_button])
display(buttons_box)

display(widgets.HTML("<h3>📝 Ответ</h3>"))
display(output_area)

display(widgets.HTML("<h3>📚 История запросов</h3>"))
display(history_area)

print("\n✅ Интерфейс загружен! Выберите вопрос или введите свой.")
print("✅ Подсчет токенов теперь ТОЧНЫЙ - данные берутся из API ответа!")